In [1]:
!wget https://data.ncl.ac.uk/ndownloader/articles/24574354/versions/1 -O ../data/AddNIST.zip
!mkdir ../data/AddNIST
!unzip ../data/AddNIST.zip -d ../data/AddNIST/
!rm ../data/AddNIST.zip

--2025-05-10 19:13:17--  https://data.ncl.ac.uk/ndownloader/articles/24574354/versions/1
Resolving data.ncl.ac.uk (data.ncl.ac.uk)... 34.255.146.38, 54.216.15.6, 52.48.143.184
Connecting to data.ncl.ac.uk (data.ncl.ac.uk)|34.255.146.38|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 659123383 (629M) [application/zip]
Saving to: ‘../data/AddNIST.zip’

../data/AddNIST.zip 100%[===================>] 628.59M  40.3MB/s    in 19s     

2025-05-10 19:13:36 (32.6 MB/s) - ‘../data/AddNIST.zip’ saved [659123383/659123383]

Archive:  ../data/AddNIST.zip
 extracting: ../data/AddNIST/metadata  
 extracting: ../data/AddNIST/test_y.npy  
 extracting: ../data/AddNIST/train_y.npy  
 extracting: ../data/AddNIST/valid_y.npy  
 extracting: ../data/AddNIST/test_x.npy  
 extracting: ../data/AddNIST/valid_x.npy  
 extracting: ../data/AddNIST/train_x.npy  
 extracting: ../data/AddNIST/README  


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from senmodel.model.utils import *
from senmodel.metrics.nonlinearity_metrics import *
from senmodel.metrics.edge_finder import *
from senmodel.metrics.train_metrics import *
from senmodel.train.train import *

In [2]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=3 * 28 * 28, output_size=20):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [3]:
hyperparams = {
    "num_epochs": 128,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.3},
    "pruning_thresholds": {"fc0": 0.1},
    "plateau_threshold": 0.05,
    "min_epochs_between_expansions": 24,
    "plateau_window_size": 5,
    "learning_rate": 5e-4,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 6000,
    "weight_decay": 1e-2,
}


"num_epochs: 128, batch_size: 256, metric: AbsGradientEdgeMetric, aggregation_mode: mean, choose_thresholds: {'fc0': 0.3}, choose_thresholds_del: {'fc0': 0.1}, threshold: 0.05, min_delta_epoch_replace: 24, window_size: 5, lr: 0.0005, delete_after: 4, task_type: classification, fully_connected: False, max_to_replace: 6000, weight_decay: 0.01"

In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

train_x = np.load("../data/AddNIST/train_x.npy")
train_y = np.load("../data/AddNIST/train_y.npy")
valid_x = np.load("../data/AddNIST/valid_x.npy")
valid_y = np.load("../data/AddNIST/valid_y.npy")
test_x = np.load('../data/AddNIST/test_x.npy')
test_y = np.load('../data/AddNIST/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 3 * 28 * 28)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 3 * 28 * 28)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 3 * 28 * 28)
test_tensor_y = torch.tensor(test_y).long()


train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)


train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False)

In [5]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [6]:
import wandb

wandb.login()

wandb: Currently logged in as: vanyamironov to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
wandb.finish()
run = wandb.init(
    project="self-expanding-nets-AddNIST",
    name=f"trash",
    config=hyperparams
)

In [8]:
import time
start_time = time.time()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, val_loader, val_loader, criterion, optimizer, hyperparams, device)

print("--- %s seconds ---" % (time.time() - start_time))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:01<00:00, 99.02it/s]


Epoch 1/128, Train Loss: 2.6247, Val Loss: 2.5187, Val Accuracy: 0.1721


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:01<00:00, 123.56it/s]


Epoch 2/128, Train Loss: 2.4368, Val Loss: 2.4740, Val Accuracy: 0.1724


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:01<00:00, 116.89it/s]


Epoch 3/128, Train Loss: 2.3815, Val Loss: 2.4371, Val Accuracy: 0.1757


... (246 outputs left out)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:26<00:00,  6.63it/s]


Epoch 127/128, Train Loss: 1.1868, Val Loss: 1.6189, Val Accuracy: 0.4725


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:26<00:00,  6.63it/s]


Epoch 128/128, Train Loss: 1.1684, Val Loss: 1.5785, Val Accuracy: 0.4777
--- 7894.4590430259705 seconds ---


In [12]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [13]:
accuracy, params

(0.4702, 9352134)

In [ ]:
torch.save(sparse_model, 'addnist.pt')

In [14]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 92.08, 94.87, 92.06, 38.06, 90.51, 93.52, 91.42, 96.60, 97.06, 97.91, 97.07, 34.17, 5],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))

styled_table

Model,Accuracy (%),Parameters
Ours,47.02,"9,352,134.0"
ResNet-18,92.08,"11,689,512.0"
AlexNet,94.87,"61,100,840.0"
VGG16,92.06,"138,357,544.0"
ConvNext,38.06,"88,591,464.0"
MNASNet,90.51,"4,383,312.0"
DenseNet,93.52,"28,681,000.0"
ResNeXt,91.42,"25,028,904.0"
PC-DARTS,96.60,nan
DrNAS,97.06,nan
